# Heritage Twin MVP - Train Gaussian Splatting từ COLMAP zip

Notebook này resume từ file `pikachu_colmap.zip` đã tạo ở notebook xử lý COLMAP.

Pipeline:

```text
upload pikachu_colmap.zip
→ unzip dataset
→ validate images + sparse/0/*.bin
→ clone/cài repo graphdeco-inria/gaussian-splatting
→ train.py bằng GPU Colab
→ zip output model để download
```

Chạy trên Google Colab GPU T4. Vào **Runtime → Change runtime type → GPU** trước khi chạy.

In [ ]:
# Cell 1 - Check GPU
!nvidia-smi

import os, sys, time, shutil, subprocess, zipfile, glob
from pathlib import Path

BASE_DIR = Path("/content/heritage_twin_train")
REPO_DIR = BASE_DIR / "gaussian-splatting"
DATA_ROOT = BASE_DIR / "datasets"
OUTPUT_ROOT = BASE_DIR / "outputs"

for p in [BASE_DIR, DATA_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

Fri Jul  3 11:15:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cấu hình

Nếu file zip của bạn không phải `pikachu_colmap.zip`, chỉ cần đổi `SCENE_NAME` và tên file upload ở cell upload. `ITERATIONS=7000` là mức hợp lý cho prototype trên T4.

In [ ]:
# Cell 2 - Config
SCENE_NAME = "pikachu"
ITERATIONS = 7000

# Training options
RESOLUTION = 1          # 1 = dùng ảnh đã undistort/resized từ COLMAP. Có thể đổi 2 nếu hết VRAM.
DATA_DEVICE = "cuda"   # nếu hết VRAM, đổi thành "cpu".
TEST_ITERATIONS = -1    # -1 để không render/test trong lúc train, giảm spike VRAM.
SAVE_ITERATIONS = ITERATIONS

SCENE_DIR = DATA_ROOT / SCENE_NAME
MODEL_DIR = OUTPUT_ROOT / f"{SCENE_NAME}_3dgs_{ITERATIONS}"

print("SCENE_DIR:", SCENE_DIR)
print("MODEL_DIR:", MODEL_DIR)

SCENE_DIR: /content/heritage_twin_train/datasets/pikachu
MODEL_DIR: /content/heritage_twin_train/outputs/pikachu_3dgs_7000


In [ ]:
# Cell 3 - Install dependencies and clone Gaussian Splatting repo
# Có thể mất 5-15 phút tùy Colab runtime.

import os, subprocess, sys
from pathlib import Path

%cd /content

if not REPO_DIR.exists():
    !git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive {REPO_DIR}
else:
    print("Repo already exists:", REPO_DIR)

%cd {REPO_DIR}

# Basic Python dependencies
!pip install -q plyfile tqdm

# Build/install CUDA extensions used by the original repo.
# Nếu cell này lỗi, restart runtime rồi chạy lại từ đầu thường xử lý được.
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn

print("Install done.")

/content
Cloning into '/content/heritage_twin_train/gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.71 MiB | 17.96 MiB/s, done.
Resolving deltas: 100% (595/595), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/heritage_twin_train/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322

## Upload `pikachu_colmap.zip`

Cell dưới dùng upload trực tiếp từ máy lên Colab. Chọn file zip bạn đã download từ notebook xử lý COLMAP.

In [ ]:
# Cell 4 - Upload COLMAP zip
from google.colab import files
from pathlib import Path
import shutil

UPLOAD_DIR = BASE_DIR / "uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Upload your COLMAP zip, e.g. pikachu_colmap.zip")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

zip_candidates = []
for filename in uploaded.keys():
    src = Path(filename)
    dst = UPLOAD_DIR / src.name
    shutil.move(str(src), str(dst))
    zip_candidates.append(dst)
    print("Uploaded:", dst, "size MB:", dst.stat().st_size / 1024 / 1024)

COLMAP_ZIP = zip_candidates[0]
print("Using COLMAP_ZIP:", COLMAP_ZIP)

Upload your COLMAP zip, e.g. pikachu_colmap.zip


Saving pikachu_colmap.zip to pikachu_colmap.zip
Uploaded: /content/heritage_twin_train/uploads/pikachu_colmap.zip size MB: 76.22488307952881
Using COLMAP_ZIP: /content/heritage_twin_train/uploads/pikachu_colmap.zip


In [ ]:
# Cell 5 - Unzip and auto-detect scene directory
import zipfile, os, shutil
from pathlib import Path

# Clean previous dataset with same scene name
if SCENE_DIR.exists():
    shutil.rmtree(SCENE_DIR)
SCENE_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = DATA_ROOT / f"_{SCENE_NAME}_extract_tmp"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", COLMAP_ZIP)
with zipfile.ZipFile(COLMAP_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Find folder that contains sparse/0/cameras.bin and either input or images.
candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    rootp = Path(root)
    if (rootp / "sparse/0/cameras.bin").exists() and (rootp / "sparse/0/images.bin").exists() and (rootp / "sparse/0/points3D.bin").exists():
        candidates.append(rootp)

if not candidates:
    print("Extracted tree preview:")
    !find {EXTRACT_DIR} -maxdepth 4 -type f | head -n 80
    raise RuntimeError("Could not find a valid COLMAP scene containing sparse/0/*.bin")

source_scene = candidates[0]
print("Detected source scene:", source_scene)

# Copy content of detected scene into SCENE_DIR
for item in source_scene.iterdir():
    dst = SCENE_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(item, dst)

print("Dataset restored to:", SCENE_DIR)
print("Top-level files/folders:")
!find {SCENE_DIR} -maxdepth 2 -type d -print

Extracting: /content/heritage_twin_train/uploads/pikachu_colmap.zip
Detected source scene: /content/heritage_twin_train/datasets/_pikachu_extract_tmp
Dataset restored to: /content/heritage_twin_train/datasets/pikachu
Top-level files/folders:
/content/heritage_twin_train/datasets/pikachu
/content/heritage_twin_train/datasets/pikachu/sparse
/content/heritage_twin_train/datasets/pikachu/sparse/0
/content/heritage_twin_train/datasets/pikachu/distorted
/content/heritage_twin_train/datasets/pikachu/distorted/sparse
/content/heritage_twin_train/datasets/pikachu/input
/content/heritage_twin_train/datasets/pikachu/stereo
/content/heritage_twin_train/datasets/pikachu/stereo/depth_maps
/content/heritage_twin_train/datasets/pikachu/stereo/consistency_graphs
/content/heritage_twin_train/datasets/pikachu/stereo/normal_maps
/content/heritage_twin_train/datasets/pikachu/images


In [ ]:
# Cell 6 - Validate dataset for train.py
from pathlib import Path
import os, glob

required = [
    SCENE_DIR / "sparse/0/cameras.bin",
    SCENE_DIR / "sparse/0/images.bin",
    SCENE_DIR / "sparse/0/points3D.bin",
]

print("Required sparse files:")
for p in required:
    print(p, "OK" if p.exists() else "MISSING")

if not all(p.exists() for p in required):
    raise RuntimeError("Missing COLMAP sparse files.")

# Graphdeco train.py expects undistorted images under images/ by default.
# convert.py usually creates images/ after undistortion. If not, fallback to input/ by copying.
images_dir = SCENE_DIR / "images"
input_dir = SCENE_DIR / "input"

if not images_dir.exists():
    if input_dir.exists():
        print("images/ missing. Copying input/ -> images/ as fallback.")
        shutil.copytree(input_dir, images_dir)
    else:
        raise RuntimeError("Neither images/ nor input/ exists.")

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(images_dir.glob(ext))

print("Image count in images/:", len(image_files))
if len(image_files) == 0:
    raise RuntimeError("No images found in images/.")

print("Dataset OK.")

Required sparse files:
/content/heritage_twin_train/datasets/pikachu/sparse/0/cameras.bin OK
/content/heritage_twin_train/datasets/pikachu/sparse/0/images.bin OK
/content/heritage_twin_train/datasets/pikachu/sparse/0/points3D.bin OK
Image count in images/: 120
Dataset OK.


In [ ]:
# Cell 7 - Train Gaussian Splatting with realtime logs
# Dự kiến với T4 + 120 frames + 7000 iterations: khoảng 20-45 phút.

import subprocess, sys, time, shutil
from pathlib import Path

%cd {REPO_DIR}

# Clean output model if exists
if MODEL_DIR.exists():
    print("Removing previous MODEL_DIR:", MODEL_DIR)
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "train.py",
    "-s", str(SCENE_DIR),
    "-m", str(MODEL_DIR),
    "--iterations", str(ITERATIONS),
    "--save_iterations", str(SAVE_ITERATIONS),
    "--test_iterations", str(TEST_ITERATIONS),
    "--resolution", str(RESOLUTION),
    "--data_device", DATA_DEVICE,
]

print("Running train.py:")
print(" ".join(cmd))
print("===== train.py realtime log =====")

start = time.time()
process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
except KeyboardInterrupt:
    print("Interrupted by user. Terminating train process...")
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
    raise

ret = process.wait()
elapsed = time.time() - start

print("===== train.py finished =====")
print("Return code:", ret)
print(f"Elapsed: {elapsed/60:.1f} minutes")

if ret != 0:
    raise RuntimeError(f"train.py failed with return code {ret}")

print("MODEL_DIR:", MODEL_DIR)

/content/heritage_twin_train/gaussian-splatting
Running train.py:
/usr/bin/python3 -u train.py -s /content/heritage_twin_train/datasets/pikachu -m /content/heritage_twin_train/outputs/pikachu_3dgs_7000 --iterations 7000 --save_iterations 7000 --test_iterations -1 --resolution 1 --data_device cuda
===== train.py realtime log =====
2026-07-03 11:25:42.277018: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/heritage_twin_train/outputs/pikachu_3dgs_7000
Output folder: /content/heritage_twin_train/outputs/pikachu_3dgs_7000 [03/07 11:25:46]

Reading camera 1/120
Reading camera 2/120
Reading camera 3/120
Reading camera 4/120
Reading camera 5/120
Reading camera 6/120
Reading camera 7/120
Reading camera 8/120
Reading camera 9/120
Readi

In [ ]:
# Cell 8 - Inspect output
from pathlib import Path

print("Model directory:", MODEL_DIR)
!find {MODEL_DIR} -maxdepth 4 -type f | sed -n '1,120p'

ply_files = list(MODEL_DIR.glob("point_cloud/iteration_*/point_cloud.ply"))
print("PLY files:")
for p in ply_files:
    print(p, "size MB:", p.stat().st_size / 1024 / 1024)

if not ply_files:
    raise RuntimeError("No point_cloud.ply found. Training may not have saved correctly.")

Model directory: /content/heritage_twin_train/outputs/pikachu_3dgs_7000
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/point_cloud/iteration_7000/point_cloud.ply
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/input.ply
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/cfg_args
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/events.out.tfevents.1783077946.f28fd08a8aaa.8068.0
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/exposure.json
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/cameras.json
PLY files:
/content/heritage_twin_train/outputs/pikachu_3dgs_7000/point_cloud/iteration_7000/point_cloud.ply size MB: 91.45041942596436


In [ ]:
# Cell 9 - Zip and download trained Gaussian model
import shutil
from google.colab import files
from pathlib import Path

zip_base = f"/content/{SCENE_NAME}_gaussian_{ITERATIONS}"
zip_path = shutil.make_archive(zip_base, "zip", MODEL_DIR)

print("Created:", zip_path)
print("Size MB:", Path(zip_path).stat().st_size / 1024 / 1024)

files.download(zip_path)

Created: /content/pikachu_gaussian_7000.zip
Size MB: 80.64465236663818


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Ghi chú

Sau khi tải được `*_gaussian_7000.zip`, bạn đã có output model của Gaussian Splatting. Bước tiếp theo mới là viewer: hoặc dùng SIBR desktop viewer, hoặc convert/sử dụng viewer web như SuperSplat / Three.js splat viewer.